In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer
%matplotlib inline

In [2]:
df = pd.read_csv("global_bleaching_environmental.csv")

/tmp/ipykernel_86407/3241338298.py:1: DtypeWarning: Columns (13,15,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("global_bleaching_environmental.csv")


In [3]:
numeric_cols = ['Latitude_Degrees', 'Longitude_Degrees', 'Distance_to_Shore',
       'Turbidity', 'Cyclone_Frequency', 'Date_Year', 'Depth_m',
       'Percent_Bleaching', 'ClimSST',
       'Temperature_Kelvin', 'Temperature_Mean', 'Temperature_Minimum',
       'Temperature_Maximum', 'Temperature_Kelvin_Standard_Deviation',
       'Windspeed', 'SSTA', 'SSTA_Standard_Deviation',
       'SSTA_Minimum', 'SSTA_Maximum', 'SSTA_Frequency',
       'SSTA_Frequency_Standard_Deviation', 'SSTA_FrequencyMax',
       'SSTA_FrequencyMean', 'SSTA_DHW', 'SSTA_DHW_Standard_Deviation',
       'SSTA_DHWMax', 'SSTA_DHWMean', 'TSA', 'TSA_Standard_Deviation',
       'TSA_Minimum', 'TSA_Maximum', 'TSA_Mean', 'TSA_Frequency',
       'TSA_Frequency_Standard_Deviation', 'TSA_FrequencyMax',
       'TSA_FrequencyMean', 'TSA_DHW', 'TSA_DHW_Standard_Deviation',
       'TSA_DHWMax', 'TSA_DHWMean',] # ssta_mean = 0

drop_columns = ['Site_ID', 'Sample_ID', 'Data_Source','Ocean_Name', 'Reef_ID', 'Realm_Name',
       'Ecoregion_Name', 'Country_Name', 'State_Island_Province_Name',
       'City_Town_Name', 'Site_Name', 'Date_Day',
       'Substrate_Name', 'Site_Comments', 'Sample_Comments',
       'Bleaching_Comments', 'Bleaching_Level', 'Percent_Cover', 'SSTA_Mean', 'Exposure',
       ] # Exposure - interesting

df.drop(columns= drop_columns, inplace= True)


for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df[df['Date_Year'] >= 1997]
df = df[df['SSTA'] > 0]
df = df.dropna(subset=['Percent_Bleaching'])

# df = df.dropna()

In [4]:
df.drop(columns= ["Date", 'Date_Year'], inplace = True)

In [9]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [5]:
X = df.drop("Percent_Bleaching", axis=1)
Y = df["Percent_Bleaching"]

In [11]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [12]:
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

# lazyPredict

In [13]:
from lazypredict.Supervised import LazyRegressor
from sklearn.model_selection import train_test_split

In [14]:
reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
models, predictions = reg.fit(X_train_imp, X_test_imp, y_train, y_test)

print(models)

                               Adjusted R-Squared     R-Squared          RMSE  \
Model                                                                           
RandomForestRegressor                7.237435e-01  7.262005e-01  1.104133e+01   
ExtraTreesRegressor                  7.223744e-01  7.248436e-01  1.106866e+01   
BaggingRegressor                     7.005578e-01  7.032210e-01  1.149534e+01   
XGBRegressor                         6.841718e-01  6.869808e-01  1.180567e+01   
HistGradientBoostingRegressor        6.303924e-01  6.336797e-01  1.277132e+01   
DecisionTreeRegressor                5.721034e-01  5.759091e-01  1.374151e+01   
KNeighborsRegressor                  5.710293e-01  5.748445e-01  1.375875e+01   
ExtraTreeRegressor                   5.163738e-01  5.206751e-01  1.460899e+01   
MLPRegressor                         5.084118e-01  5.127840e-01  1.472875e+01   
GradientBoostingRegressor            4.879457e-01  4.924999e-01  1.503222e+01   
ElasticNetCV                

In [15]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, VotingRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

# OpTuna

In [17]:
def et_objective(trial):
    model = ExtraTreesRegressor(
        n_estimators     = trial.suggest_int("n_estimators", 50, 300),
        max_depth        = trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5),
        max_features     = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1, random_state=42
    )
    pipeline = Pipeline([
        ('model',   model)
    ])
    scores = cross_val_score(pipeline, X_train_imp, y_train, cv=4, scoring='r2', n_jobs=-1)
    return scores.mean()

et_study = optuna.create_study(direction="maximize")
et_study.optimize(et_objective, n_trials=50)
print("========== ET BEST ==========")
print(et_study.best_params)
print(f"Best R²: {et_study.best_value:.4f}")

[I 2026-05-15 13:10:39,140] A new study created in memory with name: no-name-0184a14e-98d2-4f15-9080-9162649a1148
[I 2026-05-15 13:10:41,395] Trial 0 finished with value: 0.6557723852264412 and parameters: {'n_estimators': 113, 'max_depth': 37, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6557723852264412.
[I 2026-05-15 13:10:42,934] Trial 1 finished with value: 0.3475387167826699 and parameters: {'n_estimators': 190, 'max_depth': 8, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.6557723852264412.
[I 2026-05-15 13:10:44,313] Trial 2 finished with value: 0.18623192864683527 and parameters: {'n_estimators': 99, 'max_depth': 5, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.6557723852264412.
[I 2026-05-15 13:10:44,625] Trial 3 finished with value: 0.23937804413419528 and parameters: {'n_estimators': 99, 'max_depth': 6, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0

========== ET BEST ==========
{'n_estimators': 257, 'max_depth': 23, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Best R²: 0.6663


In [18]:
def rf_objective(trial):
    model = RandomForestRegressor(
        n_estimators      = trial.suggest_int("n_estimators", 50, 300),
        max_depth         = trial.suggest_int("max_depth", 5, 40),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 5),
        max_features      = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1, random_state=42
    )
    pipeline = Pipeline([
        ('model',   model)
    ])
    scores = cross_val_score(pipeline, X_train_imp, y_train, cv=4, scoring='r2', n_jobs=-1)
    return scores.mean()

rf_study = optuna.create_study(direction="maximize")
rf_study.optimize(rf_objective, n_trials=50)
print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")

[I 2026-05-15 13:11:46,734] A new study created in memory with name: no-name-8515b64c-a4d1-4b00-9131-d31cc25c9b3b
[I 2026-05-15 13:11:47,427] Trial 0 finished with value: 0.5300556155893033 and parameters: {'n_estimators': 77, 'max_depth': 7, 'min_samples_split': 3, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5300556155893033.
[I 2026-05-15 13:11:50,422] Trial 1 finished with value: 0.6336820510715726 and parameters: {'n_estimators': 241, 'max_depth': 35, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 1 with value: 0.6336820510715726.
[I 2026-05-15 13:11:51,606] Trial 2 finished with value: 0.596407325934879 and parameters: {'n_estimators': 131, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 1 with value: 0.6336820510715726.
[I 2026-05-15 13:11:52,518] Trial 3 finished with value: 0.5316557820011343 and parameters: {'n_estimators': 106, 'max_depth': 7, 'min_

========== RF BEST ==========
{'n_estimators': 256, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Best R²: 0.6712


In [19]:
def xgb_objective(trial):
    model = xgb.XGBRegressor(
        n_estimators     = trial.suggest_int("n_estimators", 50, 400),
        learning_rate    = trial.suggest_float("learning_rate", 0.005, 0.1),
        max_depth        = trial.suggest_int("max_depth", 3, 15),
        subsample        = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        n_jobs=-1, random_state=42
    )
    pipeline = Pipeline([
        ('model',   model)
    ])
    scores = cross_val_score(pipeline, X_train_imp, y_train, cv=4, scoring='r2', n_jobs=-1)
    return scores.mean()

xgb_study = optuna.create_study(direction="maximize")
xgb_study.optimize(xgb_objective, n_trials=50)
print("========== XGB BEST ==========")
print(xgb_study.best_params)
print(f"Best R²: {xgb_study.best_value:.4f}")

[I 2026-05-15 13:13:59,444] A new study created in memory with name: no-name-714ae0f8-7e43-4d9e-a058-3b5b4f9c34b9
[I 2026-05-15 13:14:00,367] Trial 0 finished with value: 0.5193985111545959 and parameters: {'n_estimators': 152, 'learning_rate': 0.020113444590040844, 'max_depth': 4, 'subsample': 0.8597806629793422, 'colsample_bytree': 0.6880081339413908}. Best is trial 0 with value: 0.5193985111545959.
[I 2026-05-15 13:14:01,167] Trial 1 finished with value: 0.5727549033188923 and parameters: {'n_estimators': 153, 'learning_rate': 0.053240913124636274, 'max_depth': 4, 'subsample': 0.9958718742220305, 'colsample_bytree': 0.5535845459259139}. Best is trial 1 with value: 0.5727549033188923.
[I 2026-05-15 13:14:12,472] Trial 2 finished with value: 0.6865408078742086 and parameters: {'n_estimators': 99, 'learning_rate': 0.052583508438004474, 'max_depth': 14, 'subsample': 0.7904936077407643, 'colsample_bytree': 0.8482195837247732}. Best is trial 2 with value: 0.6865408078742086.
[I 2026-05-15

========== XGB BEST ==========
{'n_estimators': 400, 'learning_rate': 0.02310955881244883, 'max_depth': 11, 'subsample': 0.9791745793947237, 'colsample_bytree': 0.5273671772623845}
Best R²: 0.6962


# Start

In [20]:
et_model = ExtraTreesRegressor(
    **et_study.best_params,
    n_jobs=-1,
    random_state=42
)
# {'n_estimators': 224, 'max_depth': 29, 'min_samples_leaf': 1, 'max_features': 'sqrt'}

rf_model = RandomForestRegressor(
    **rf_study.best_params,
    n_jobs=-1,
    random_state=42
)
# {'n_estimators': 135, 'max_depth': 40, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt'}

xgb_model = xgb.XGBRegressor(
    **xgb_study.best_params,
    n_jobs=-1,
    random_state=42
)
# {'n_estimators': 337, 'learning_rate': 0.028625556453099745, 'max_depth': 11, 'subsample': 0.9048951178914527,
#   'colsample_bytree': 0.7006625246510426}



In [21]:
voting_model = VotingRegressor([
    ('et',  et_model),
    ('rf',  rf_model),
    ('xgb', xgb_model)
])

# Train all 4
et_model.fit(X_train_imp, y_train)
rf_model.fit(X_train_imp, y_train)
xgb_model.fit(X_train_imp, y_train)
voting_model.fit(X_train_imp, y_train)

# Evaluate
models = {
    'ExtraTrees':        et_model,
    'RandomForest':      rf_model,
    'XGBoost':           xgb_model,
    'Voting (Ensemble)': voting_model
}

print("=" * 65)
print("BASELINE RESULTS (single seed = 42)")
print("=" * 65)
print(f"{'Model':<22}{'R²':>10}{'RMSE':>12}{'MAE':>10}{'Acc%':>10}")
print("-" * 65)
for name, model in models.items():
    y_pred = model.predict(X_test_imp)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    print(f"{name:<22}{r2:>10.4f}{rmse:>12.4f}{mae:>10.4f}{100-mae:>10.2f}")
print("-" * 65)
print(f"{'Paper baseline:':<22}{0.6395:>10.4f}{12.9268:>12.4f}{'—':>10}{93.88:>10.2f}")
print("=" * 65)

BASELINE RESULTS (single seed = 42)
Model                         R²        RMSE       MAE      Acc%
-----------------------------------------------------------------
ExtraTrees                0.7102     11.3593    5.0243     94.98
RandomForest              0.7028     11.5027    5.8754     94.12
XGBoost                   0.7359     10.8448    5.1674     94.83
Voting (Ensemble)         0.7268     11.0296    5.2997     94.70
-----------------------------------------------------------------
Paper baseline:           0.6395     12.9268         —     93.88


# K-Fold

In [24]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline

cv = KFold(n_splits=5, shuffle=True, random_state=42)   # ← define cv

def make_pipeline(model):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        # ('scaler',  StandardScaler()),
        # ('pca',     PCA(n_components=0.95)),
        ('model',   model)
    ])

cv_models = {
    'ExtraTrees':        make_pipeline(et_model),
    'RandomForest':      make_pipeline(rf_model),
    'XGBoost':           make_pipeline(xgb_model),
    'Voting (Ensemble)': make_pipeline(voting_model)
}

print("=" * 70)
print("K-Fold Cross Validation Summary (k=5)")
print("=" * 70)
print(f"{'Model':<22}{'Mean R²':>14}{'Mean RMSE':>14}{'Mean MAE':>14}")
print("-" * 70)

for name, model in cv_models.items():
    cv_results = cross_validate(
        model, X, Y, cv=cv,                        
        scoring={'r2': 'r2',
                 'neg_mse': 'neg_mean_squared_error',
                 'neg_mae': 'neg_mean_absolute_error'},
        n_jobs=-1
    )
    r2_mean   = cv_results['test_r2'].mean()
    rmse_mean = np.sqrt(-cv_results['test_neg_mse']).mean()
    mae_mean  = (-cv_results['test_neg_mae']).mean()
    print(f"{name:<22}{r2_mean:>14.4f}{rmse_mean:>14.4f}{mae_mean:>14.4f}")

print("=" * 70)

K-Fold Cross Validation Summary (k=5)
Model                        Mean R²     Mean RMSE      Mean MAE
----------------------------------------------------------------------


ExtraTrees                    0.7018       11.7605        5.2458
RandomForest                  0.6995       11.8053        6.0467
XGBoost                       0.7282       11.2289        5.3945
Voting (Ensemble)             0.7197       11.4015        5.5051
